# Day 5 — Memory and RAG
Run in Colab. Upload faq.json and tests.csv to /content first. Add GROQ_API_KEY to Secrets. Uses Groq GPT-OSS 120B; no paid embedding API. Run cells individually.

In [ ]:
%pip install -q langchain langchain-groq langgraph chromadb gspread pandas

In [ ]:
from google.colab import userdata
from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-120b", api_key=userdata.get("GROQ_API_KEY").strip(), temperature=0.2, max_retries=0)

## Day 4 prerequisites — tools and optional Sheets

In [ ]:
from langchain.tools import tool
MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}
HOURS = dict.fromkeys(["monday", "tuesday", "wednesday", "thursday", "friday", "saturday"], "9 am to 8 pm")
HOURS["sunday"] = "10 am to 6 pm"

@tool
def get_price(item: str, qty: int = 1) -> str:
    """Calculate total rupee price for a menu item, e.g. chocolate cake, qty 2."""
    item = item.lower().strip()
    if qty < 1:
        return "Quantity must be at least 1."
    if item not in MENU:
        return f"Not on the menu. Available items: {', '.join(MENU)}"
    return f"{qty} x {item} = Rs {MENU[item] * qty}"

@tool
def is_available(item: str) -> str:
    """Check whether an item, e.g. cupcake, is offered on the menu, not live stock."""
    return f"{item} is on the menu; stock is not verified." if item.lower().strip() in MENU else f"{item} is not on the menu."

@tool
def is_open(day: str) -> str:
    """Look up opening hours for a weekday, e.g. sunday."""
    return HOURS.get(day.lower().strip(), "Invalid weekday")

print(get_price.invoke({"item": "chocolate cake", "qty": 2}))
print(is_available.invoke({"item": "cupcake"}))
print(is_open.invoke({"day": "sunday"}))

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
class Order(BaseModel):
    """A bakery pickup order."""
    name: str = Field(min_length=1)
    item: Literal["plum cake", "chocolate cake", "cupcake"]
    qty: int = Field(ge=1)
    date: str = Field(min_length=1, description="Pickup date as stated")
    eggless: bool

In [ ]:
ENABLE_SHEETS = False
ws = None
if ENABLE_SHEETS:
    from google.colab import auth
    auth.authenticate_user()
    import gspread
    from google.auth import default
    creds, _ = default()
    gc = gspread.authorize(creds)
    try:
        sh = gc.open("Sweet Crumbs Orders - Day 4 Test")
    except gspread.SpreadsheetNotFound:
        sh = gc.create("Sweet Crumbs Orders - Day 4 Test")
    ws = sh.sheet1
    if not ws.row_values(1):
        ws.append_row(["name", "item", "qty", "date", "eggless", "total"])
    print("Sheet ready:", sh.url)

In [ ]:
@tool
def log_order(name: str, item: str, qty: int, date: str, eggless: bool) -> str:
    """Save a confirmed pickup order with customer name, menu item, positive quantity, pickup date and eggless preference."""
    try:
        order = Order(name=name.strip(), item=item.lower().strip(), qty=qty, date=date.strip(), eggless=eggless)
        if order.date.lower() in {"unknown", "none", "not specified"}:
            return "Ask for pickup date."
    except ValueError as e:
        return f"Invalid order: {e}"
    if ws is None:
        return "Order not saved: enable Sheets first."
    total = MENU[order.item] * order.qty
    try:
        ws.append_row([order.name, order.item, order.qty, order.date, "yes" if order.eggless else "no", total], value_input_option="RAW")
    except Exception as e:
        return f"Save not confirmed: {type(e).__name__}. Check sheet before retrying."
    return f"Saved: {order.qty} x {order.item} for {order.name} on {order.date}, Rs {total}."


## Task 1 — Knowledge (19 facts, one per chunk)

In [ ]:
import json, chromadb
from pathlib import Path
FAQS = json.loads(Path('faq.json').read_text())
client = chromadb.Client()
faq = client.get_or_create_collection('sweet_crumbs_faq')
faq.upsert(ids=list(FAQS), documents=list(FAQS.values()))
print(faq.count(), 'facts stored')
print(faq.query(query_texts=['Can you send cake to my house?'], n_results=3)['documents'][0])

## Task 2 — Search tool

In [ ]:
@tool
def search_faq(question: str) -> str:
    """Search delivery, payment, advance, cancellations, custom cakes, notice, eggless, allergens, storage, messages, candles, discounts and pickup policies."""
    r = faq.query(query_texts=[question], n_results=4)
    return '\n'.join(f'[{i}] {doc}' for i, doc in zip(r['ids'][0], r['documents'][0]))
print(search_faq.invoke({'question':'How early should I order a photo cake?'}))

## Task 3 — Agent and separate customer threads

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from uuid import uuid4
SYSTEM = """You are Sweet Crumbs Home Bakery's friendly assistant. Use get_price for prices and is_open for hours. Use search_faq for policy questions and answer only from retrieved facts. If facts do not support an answer, say I'll check with the owner. Do not assume a delivery location is within 5 km. Never invent stock or allergy guarantees. For unsupported menu items say I'll check with the owner. Save only explicitly confirmed orders with name, item, positive quantity, pickup date and eggless preference; never claim saved unless log_order confirms. Reply under 60 words in English or Telugu."""
agent = create_agent(model=llm, tools=[get_price,is_open,log_order,search_faq], system_prompt=SYSTEM, checkpointer=InMemorySaver())
def chat(thread_id, text):
    result = agent.invoke({'messages':[{'role':'user','content':text}]}, config={'configurable':{'thread_id':thread_id},'recursion_limit':10})
    return result['messages'][-1].content


In [ ]:
session = uuid4().hex
for customer, question in [('ravi', "Hi, I'm Ravi. Do you deliver? My order is about Rs 1,300."), ('sravani', "Hi, I'm Sravani. Can I pay by UPI?"), ('ravi', "What's my name, and will I pay for delivery?"), ('sravani', "What is my name?")]:
    print(customer, ':', chat(session+customer, question))

## Task 4 — Break it: unsupported knowledge

In [ ]:
print(chat(uuid4().hex, 'Do you sell biryani?'))
print(chat(uuid4().hex, 'Do you offer international shipping?'))
# Expected: check with the owner, without inventing a policy.

## Task 5 — Sign-off
Run all 15 tests once. Errors are reported separately; no invented pass rate. Resume by rerunning this cell; completed results are kept in memory.

In [ ]:
import pandas as pd
import re
tests = pd.read_csv('tests.csv')
if 'test_results' not in globals(): test_results = []
def normalize(s): return re.sub(r'\s+', ' ', str(s).lower().replace(',', '')).strip()
for i in range(len(test_results), len(tests)):
    row = tests.iloc[i]
    try:
        answer = chat(uuid4().hex, row.question)
    except Exception as e:
        print('Stopped at test', i+1, type(e).__name__, str(e)[:300]); break
    passed = normalize(row.must_include) in normalize(answer)
    test_results.append({'question':row.question,'expected':row.must_include,'answer':answer,'result':'PASS' if passed else 'FAIL'})
    print(i+1, test_results[-1]['result'], answer)
report = pd.DataFrame(test_results)
report.to_csv('test_results.csv',index=False)
if len(report):
    passed = (report.result == 'PASS').sum()
    print(f'Passed: {passed}/{len(tests)}; completed: {len(report)}/{len(tests)}')
    if len(report)==len(tests): print(f'Final pass rate: {100*passed/len(tests):.1f}%')
    else: print('Incomplete: resume after quota resets.')

## Before Day 6
Customer request → classify → retrieve policy or check price/hours → answer; for orders → collect details → confirm → save → report result.
Memory is in-process only; it disappears when Colab restarts.